# TV2 - DeBERTa Hyperparameter Tuning

Notebook này fine-tune DeBERTa-v3-base theo chiến lược tuning ít nhưng có chủ đích:

1. Stage 1: tune learning rate (`lr_backbone`, `lr_head`).
2. Stage 2: tune input length and pooling (`max_length`, `pooling`).
3. Stage 3: tune regularization/scheduler (`dropout`, `scheduler_type`, `label_smoothing`, `use_class_weights`).

Notebook dùng Stratified K-Fold đã tạo ở TV1: `fold=0` làm validation, các fold còn lại làm train. Không tạo test split riêng trong notebook này.


## 0. Imports & global config


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Dict, Iterable, List, Optional
import gc
import json
import os
import random
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoConfig,
    AutoModel,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle/working').exists()
ROOT = Path('/kaggle/working') if IS_KAGGLE else Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
if not IS_KAGGLE and str(Path('..').resolve()) not in sys.path:
    sys.path.append(str(Path('..').resolve()))

try:
    from src.modeling import DeBERTaDetector
except Exception:
    class MeanPooling(nn.Module):
        def forward(self, hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
            mask = attention_mask.unsqueeze(-1).float()
            summed = (hidden_state * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            return summed / counts

    class DeBERTaDetector(nn.Module):
        def __init__(self, model_name: str, num_labels: int = 2, dropout: float = 0.1, pooling: str = 'mean'):
            super().__init__()
            if pooling not in {'mean', 'cls'}:
                raise ValueError("pooling must be either 'mean' or 'cls'")
            self.pooling = pooling
            self.config = AutoConfig.from_pretrained(
                model_name,
                num_labels=num_labels,
                hidden_dropout_prob=dropout,
                attention_probs_dropout_prob=dropout,
            )
            self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
            self.mean_pooler = MeanPooling()
            self.dropout = nn.Dropout(dropout)
            self.classifier = nn.Linear(self.config.hidden_size, num_labels)
            nn.init.normal_(self.classifier.weight, mean=0.0, std=self.config.initializer_range)
            nn.init.zeros_(self.classifier.bias)

        def forward(self, input_ids, attention_mask, token_type_ids=None):
            outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
            pooled = outputs.last_hidden_state[:, 0] if self.pooling == 'cls' else self.mean_pooler(outputs.last_hidden_state, attention_mask)
            return self.classifier(self.dropout(pooled))

DATA_DIR = ROOT / 'data'
OUT_DIR = ROOT / 'outputs' / 'tv2_deberta_tuning'
CKPT_DIR = OUT_DIR / 'checkpoints'
LOG_DIR = OUT_DIR / 'logs'
SUMMARY_PATH = OUT_DIR / 'experiment_summary.csv'
BEST_CONFIG_PATH = OUT_DIR / 'best_config.json'
PROMOTED_CKPT_PATH = ROOT / 'outputs' / 'checkpoints' / 'deberta_tuned_best.pt'

for path in [OUT_DIR, CKPT_DIR, LOG_DIR, PROMOTED_CKPT_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEFAULT_NUM_WORKERS = 2 if IS_KAGGLE else (0 if os.name == 'nt' else 2)
USE_DATA_PARALLEL = torch.cuda.device_count() > 1

# On Kaggle GPU this is enabled by default. Locally it remains disabled for safety.
RUN_TUNING = IS_KAGGLE and torch.cuda.is_available()

# Options: 'minimal6' or 'full11'. Keep minimal6 for a reasonable Kaggle runtime.
TUNING_PLAN = 'minimal6'
INCLUDE_LLRD_RUN = False


def find_first_file(file_names: List[str]) -> Optional[Path]:
    search_roots = [DATA_DIR]
    if IS_KAGGLE:
        search_roots.append(Path('/kaggle/input'))
    for root in search_roots:
        if not root.exists():
            continue
        for file_name in file_names:
            direct = root / file_name
            if direct.exists():
                return direct
        for file_name in file_names:
            matches = sorted(root.rglob(file_name))
            if matches:
                return matches[0]
    return None


def find_deberta_model(default_model: str = 'microsoft/deberta-v3-base') -> str:
    env_path = os.environ.get('DEBERTA_MODEL_PATH')
    if env_path and Path(env_path).exists():
        return env_path
    if IS_KAGGLE and Path('/kaggle/input').exists():
        candidates = []
        for config_path in Path('/kaggle/input').rglob('config.json'):
            model_dir = config_path.parent
            name = str(model_dir).lower()
            has_weights = any((model_dir / fname).exists() for fname in ['pytorch_model.bin', 'model.safetensors'])
            if 'deberta' in name and has_weights:
                candidates.append(model_dir)
        if candidates:
            return str(sorted(candidates, key=lambda p: len(str(p)))[0])
    return default_model


def get_model_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()


def unwrap_model(model: nn.Module) -> nn.Module:
    return model.module if isinstance(model, nn.DataParallel) else model


def force_trainable_fp32(model: nn.Module) -> nn.Module:
    # Some Kaggle offline model snapshots store weights in fp16.
    # GradScaler expects FP32 trainable parameters and uses autocast for mixed precision.
    return model.float()

TRAIN_CLEAN_PATH = find_first_file(['train_clean.pkl'])
TRAIN_CSV_PATH = find_first_file(['train_v2_drcat_02.csv', 'train.csv'])
MODEL_NAME_OR_PATH = find_deberta_model()

print(f'Kaggle : {IS_KAGGLE}')
print(f'ROOT   : {ROOT}')
print(f'Device : {DEVICE}')
print(f'GPUs   : {torch.cuda.device_count()} | DataParallel={USE_DATA_PARALLEL}')
print(f'Plan   : {TUNING_PLAN}')
print(f'Run    : {RUN_TUNING}')
print(f'Workers: {DEFAULT_NUM_WORKERS}')
print(f'Data pkl: {TRAIN_CLEAN_PATH}')
print(f'Data csv: {TRAIN_CSV_PATH}')
print(f'Model  : {MODEL_NAME_OR_PATH}')


## 1. Reproducibility helpers


In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)


## 2. Experiment config

Baseline default config: `max_length=512`, `pooling='mean'`, `lr_backbone=2e-5`, `lr_head=1e-4`, `dropout=0.1`, `scheduler_type='linear'`.


In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    stage: str

    # Model
    model_name: str = MODEL_NAME_OR_PATH
    max_length: int = 512
    num_labels: int = 2
    pooling: str = 'mean'
    dropout: float = 0.1

    # Training
    epochs: int = 4
    batch_size: int = 8
    valid_batch_size: int = 16
    grad_accum: int = 4
    max_grad_norm: float = 1.0
    num_workers: int = DEFAULT_NUM_WORKERS

    # Optimizer / Scheduler
    lr_backbone: float = 2e-5
    lr_head: float = 1e-4
    llrd_factor: float = 0.9
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    scheduler_type: str = 'linear'  # 'linear' or 'cosine'
    eps: float = 1e-6
    beta1: float = 0.9
    beta2: float = 0.999

    # AMP / Early stopping
    use_amp: bool = False
    patience: int = 2
    min_delta: float = 1e-4
    selection_auc_tolerance: float = 5e-4
    selection_f1_tolerance: float = 1e-4

    # Metric / imbalance
    primary_metric: str = 'auc'
    label_smoothing: float = 0.0
    use_class_weights: bool = False

BASE_CFG = ExperimentConfig(name='run01_lr2e5_head1e4_512_mean', stage='stage1_lr')
asdict(BASE_CFG)


## 3. Load K-Fold data

Notebook dùng `fold=0` làm validation. Đây là cùng quy ước với Task 3 và Task 4.


In [ ]:
def load_training_dataframe() -> pd.DataFrame:
    if TRAIN_CLEAN_PATH is not None and TRAIN_CLEAN_PATH.exists():
        print(f'Loading cleaned data: {TRAIN_CLEAN_PATH}')
        return pd.read_pickle(TRAIN_CLEAN_PATH)
    if TRAIN_CSV_PATH is not None and TRAIN_CSV_PATH.exists():
        print(f'Loading CSV data: {TRAIN_CSV_PATH}')
        return pd.read_csv(TRAIN_CSV_PATH)
    raise FileNotFoundError(
        'Could not find train_clean.pkl or DAIGT CSV. '
        'On Kaggle, add the dataset that contains train_clean.pkl or train_v2_drcat_02.csv.'
    )


df = load_training_dataframe()
text_col = 'text_clean' if 'text_clean' in df.columns else 'text'
if text_col not in df.columns:
    raise ValueError("Could not find a text column. Expected 'text_clean' or 'text'.")
if 'text_clean' not in df.columns:
    df['text_clean'] = df[text_col].astype(str)
    text_col = 'text_clean'

if 'label' not in df.columns:
    raise ValueError("Could not find label column 'label'.")


def normalize_for_split(series: pd.Series) -> pd.Series:
    return (
        series.fillna('')
        .astype(str)
        .str.lower()
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )


df = df.copy().reset_index(drop=True)
df['_normalized_text_for_split'] = normalize_for_split(df[text_col])
normalized_duplicate_count = int(df['_normalized_text_for_split'].duplicated().sum())
if normalized_duplicate_count > 0:
    print(f'Removing {normalized_duplicate_count} normalized duplicate rows before splitting.')
    df = df.drop_duplicates('_normalized_text_for_split', keep='first').reset_index(drop=True)
    # Rebuild split/fold after duplicate removal so no duplicate can cross train/valid/test.
    df = df.drop(columns=['split', 'fold'], errors='ignore')

VAL_FOLD = 0
TEST_SIZE = 0.10

# Hold out test first. Use folds only within train_val for tuning/training.
if 'split' not in df.columns:
    print(f'No split column found. Creating {TEST_SIZE:.0%} stratified held-out test split.')
    df = df.copy().reset_index(drop=True)
    df['split'] = 'train_val'
    train_val_idx, test_idx = train_test_split(
        df.index,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=df['label'],
    )
    df.loc[test_idx, 'split'] = 'test'
    df['fold'] = -1
else:
    df = df.copy().reset_index(drop=True)
    if 'fold' not in df.columns:
        df['fold'] = -1

train_val_mask = df['split'].eq('train_val')
if (df.loc[train_val_mask, 'fold'] < 0).any():
    print('Creating StratifiedKFold folds on train_val rows only.')
    df.loc[:, 'fold'] = -1
    train_val_idx = df.index[train_val_mask].to_numpy()
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold_id, (_, val_pos) in enumerate(skf.split(df.loc[train_val_idx], df.loc[train_val_idx, 'label'])):
        df.loc[train_val_idx[val_pos], 'fold'] = fold_id

df.loc[df['split'].eq('test'), 'fold'] = -1

train_val_df = df[df['split'] == 'train_val'].reset_index(drop=True)
train_df = train_val_df[train_val_df['fold'] != VAL_FOLD].reset_index(drop=True)
valid_df = train_val_df[train_val_df['fold'] == VAL_FOLD].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)

print(f'Train: {train_df.shape} | Valid: {valid_df.shape} | Test: {test_df.shape}')
print('Train label distribution:')
print(train_df['label'].value_counts(normalize=True).sort_index())
print('Valid label distribution:')
print(valid_df['label'].value_counts(normalize=True).sort_index())
print('Test label distribution:')
print(test_df['label'].value_counts(normalize=True).sort_index())


def count_overlap(left: pd.DataFrame, right: pd.DataFrame) -> int:
    return len(set(left['_normalized_text_for_split']) & set(right['_normalized_text_for_split']))


leakage_df = pd.DataFrame(
    [
        {'check': 'train_vs_valid', 'overlap': count_overlap(train_df, valid_df)},
        {'check': 'train_vs_test', 'overlap': count_overlap(train_df, test_df)},
        {'check': 'valid_vs_test', 'overlap': count_overlap(valid_df, test_df)},
    ]
)
display(leakage_df)
if int(leakage_df['overlap'].sum()) != 0:
    raise ValueError('Text leakage detected across train/valid/test splits. Rebuild data split before training.')


## 4. Dataset and dataloaders


In [ ]:
class AITextDataset(Dataset):
    def __init__(self, texts: Iterable[str], labels: Iterable[int], tokenizer, max_length: int):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
            return_attention_mask=True,
        )
        item = {key: value.squeeze(0) for key, value in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def make_loaders(cfg: ExperimentConfig):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    train_ds = AITextDataset(train_df[text_col].tolist(), train_df['label'].tolist(), tokenizer, cfg.max_length)
    valid_ds = AITextDataset(valid_df[text_col].tolist(), valid_df['label'].tolist(), tokenizer, cfg.max_length)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=cfg.valid_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )
    return tokenizer, train_loader, valid_loader


## 5. Optimizer groups with LLRD


In [ ]:
def split_decay_params(named_params, no_decay):
    decay = [p for n, p in named_params if p.requires_grad and not any(nd in n for nd in no_decay)]
    no_decay_params = [p for n, p in named_params if p.requires_grad and any(nd in n for nd in no_decay)]
    return decay, no_decay_params


def get_optimizer_groups(model: nn.Module, cfg: ExperimentConfig) -> List[Dict]:
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    groups = []

    head_named = list(model.classifier.named_parameters())
    decay, no_decay_params = split_decay_params(head_named, no_decay)
    if decay:
        groups.append({'params': decay, 'lr': cfg.lr_head, 'weight_decay': cfg.weight_decay})
    if no_decay_params:
        groups.append({'params': no_decay_params, 'lr': cfg.lr_head, 'weight_decay': 0.0})

    n_layers = model.config.num_hidden_layers
    for layer_idx in range(n_layers - 1, -1, -1):
        depth = n_layers - 1 - layer_idx
        lr_cur = cfg.lr_backbone * (cfg.llrd_factor ** depth)
        named = list(model.backbone.encoder.layer[layer_idx].named_parameters())
        decay, no_decay_params = split_decay_params(named, no_decay)
        if decay:
            groups.append({'params': decay, 'lr': lr_cur, 'weight_decay': cfg.weight_decay})
        if no_decay_params:
            groups.append({'params': no_decay_params, 'lr': lr_cur, 'weight_decay': 0.0})

    lr_emb = cfg.lr_backbone * (cfg.llrd_factor ** n_layers)
    emb_named = list(model.backbone.embeddings.named_parameters())
    decay, no_decay_params = split_decay_params(emb_named, no_decay)
    if decay:
        groups.append({'params': decay, 'lr': lr_emb, 'weight_decay': cfg.weight_decay})
    if no_decay_params:
        groups.append({'params': no_decay_params, 'lr': lr_emb, 'weight_decay': 0.0})

    return groups


## 6. Loss, metrics, scheduler


In [ ]:
def make_class_weights(labels: Iterable[int], device: torch.device) -> Optional[torch.Tensor]:
    labels = np.asarray(list(labels)).astype(int)
    counts = np.bincount(labels, minlength=2)
    if (counts == 0).any():
        return None
    weights = counts.sum() / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32, device=device)


def make_criterion(cfg: ExperimentConfig) -> nn.Module:
    class_weights = make_class_weights(train_df['label'], DEVICE) if cfg.use_class_weights else None
    return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg.label_smoothing)


def sanitize_probabilities(probs) -> np.ndarray:
    probs = np.asarray(probs, dtype=np.float64)
    if not np.isfinite(probs).all():
        n_bad = int((~np.isfinite(probs)).sum())
        print(f'[WARN] Non-finite probabilities detected: {n_bad}. Replacing with neutral/clipped values.')
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)
    return np.clip(probs, 0.0, 1.0)


def compute_metrics(labels, probs, threshold: float = 0.5) -> Dict[str, float]:
    labels = np.asarray(labels).astype(int)
    probs = sanitize_probabilities(probs)
    preds = (probs >= threshold).astype(int)

    if len(np.unique(labels)) < 2:
        roc_auc = np.nan
        pr_auc = np.nan
    else:
        roc_auc = roc_auc_score(labels, probs)
        pr_auc = average_precision_score(labels, probs)

    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
    }


def make_scheduler(cfg: ExperimentConfig, optimizer, total_steps: int):
    warmup_steps = int(total_steps * cfg.warmup_ratio)
    if cfg.scheduler_type == 'linear':
        return get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    if cfg.scheduler_type == 'cosine':
        return get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    raise ValueError(f'Unknown scheduler_type: {cfg.scheduler_type}')


## 7. Train and evaluate functions


In [ ]:
def assert_finite_tensor(tensor: torch.Tensor, name: str, cfg_name: str, step: Optional[int] = None) -> None:
    if not torch.isfinite(tensor).all():
        where = f' at step {step}' if step is not None else ''
        raise FloatingPointError(f'Non-finite {name} detected in {cfg_name}{where}.')


def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, cfg: ExperimentConfig, epoch: int):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'{cfg.name} | epoch {epoch+1} train', leave=False, ncols=100)
    for step, batch in enumerate(pbar):
        labels = batch.pop('labels').to(DEVICE)
        batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

        with autocast(enabled=cfg.use_amp and torch.cuda.is_available()):
            logits = model(**batch)
            assert_finite_tensor(logits, 'train logits', cfg.name, step)
            loss = criterion(logits, labels) / cfg.grad_accum
            assert_finite_tensor(loss, 'train loss', cfg.name, step)

        scaler.scale(loss).backward()

        if (step + 1) % cfg.grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(unwrap_model(model).parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * cfg.grad_accum
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += len(labels)
        pbar.set_postfix(loss=f'{total_loss/(step+1):.4f}', acc=f'{correct/max(total,1):.4f}')

    return {'loss': total_loss / max(len(loader), 1), 'accuracy': correct / max(total, 1)}


@torch.no_grad()
def evaluate(model, loader, criterion, cfg: ExperimentConfig):
    model.eval()
    total_loss = 0.0
    probs = []
    labels_all = []

    pbar = tqdm(loader, desc=f'{cfg.name} valid', leave=False, ncols=100)
    for step, batch in enumerate(pbar):
        labels = batch.pop('labels').to(DEVICE)
        batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
        with autocast(enabled=cfg.use_amp and torch.cuda.is_available()):
            logits = model(**batch)
            assert_finite_tensor(logits, 'valid logits', cfg.name, step)
            loss = criterion(logits, labels)
            assert_finite_tensor(loss, 'valid loss', cfg.name, step)
        prob_ai = torch.softmax(logits.float(), dim=-1)[:, 1]
        assert_finite_tensor(prob_ai, 'valid probabilities', cfg.name, step)

        total_loss += loss.item()
        probs.extend(prob_ai.detach().cpu().numpy().tolist())
        labels_all.extend(labels.detach().cpu().numpy().tolist())

    probs = sanitize_probabilities(probs)
    metrics = compute_metrics(labels_all, probs)
    metrics['loss'] = total_loss / max(len(loader), 1)
    metrics['probs'] = np.asarray(probs)
    metrics['labels'] = np.asarray(labels_all)
    return metrics


## 8. Experiment runner

Mỗi run lưu:

- checkpoint tốt nhất theo validation ROC-AUC,
- log từng epoch,
- summary cấu hình + metric tốt nhất.


In [ ]:
def checkpoint_path_for(cfg: ExperimentConfig) -> Path:
    return CKPT_DIR / f'{cfg.name}_best.pt'


def is_better_checkpoint(row: Dict, best_row: Dict | None, cfg: ExperimentConfig) -> bool:
    if best_row is None:
        return True

    auc_gain = row['val_auc'] - best_row['val_auc']
    if auc_gain > cfg.selection_auc_tolerance:
        return True

    # When AUC is effectively tied, prefer the checkpoint that works better at the
    # operating threshold and is less over-confident. This avoids replacing a good
    # epoch just because AUC improves by a tiny amount while F1/loss collapse.
    if abs(auc_gain) <= cfg.selection_auc_tolerance:
        f1_gain = row['val_f1'] - best_row['val_f1']
        if f1_gain > cfg.selection_f1_tolerance:
            return True
        if abs(f1_gain) <= cfg.selection_f1_tolerance and row['val_loss'] < best_row['val_loss']:
            return True

    return False


def run_experiment(cfg: ExperimentConfig) -> Dict:
    seed_everything(SEED)
    print('\n' + '=' * 80)
    print(f'Running: {cfg.name}')
    print(json.dumps(asdict(cfg), indent=2))
    print('=' * 80)

    _, train_loader, valid_loader = make_loaders(cfg)
    base_model = DeBERTaDetector(
        model_name=cfg.model_name,
        num_labels=cfg.num_labels,
        dropout=cfg.dropout,
        pooling=cfg.pooling,
    ).to(DEVICE)
    base_model = force_trainable_fp32(base_model)

    optimizer = torch.optim.AdamW(
        get_optimizer_groups(base_model, cfg),
        betas=(cfg.beta1, cfg.beta2),
        eps=cfg.eps,
    )
    model = nn.DataParallel(base_model) if USE_DATA_PARALLEL else base_model
    if USE_DATA_PARALLEL:
        print(f'Using DataParallel on {torch.cuda.device_count()} GPUs')
    total_steps = max(1, (len(train_loader) // cfg.grad_accum) * cfg.epochs)
    scheduler = make_scheduler(cfg, optimizer, total_steps)
    scaler = GradScaler(enabled=cfg.use_amp and torch.cuda.is_available())
    print(f'AMP enabled: {cfg.use_amp and torch.cuda.is_available()}')
    criterion = make_criterion(cfg)

    ckpt_path = checkpoint_path_for(cfg)
    history = []
    best_auc = -np.inf
    best_row = None
    best_epoch = -1
    patience_counter = 0
    start = time.time()

    for epoch in range(cfg.epochs):
        train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, criterion, cfg, epoch)
        valid_metrics = evaluate(model, valid_loader, criterion, cfg)

        row = {
            'run_name': cfg.name,
            'stage': cfg.stage,
            'epoch': epoch + 1,
            'train_loss': train_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'val_loss': valid_metrics['loss'],
            'val_accuracy': valid_metrics['accuracy'],
            'val_precision': valid_metrics['precision'],
            'val_recall': valid_metrics['recall'],
            'val_f1': valid_metrics['f1'],
            'val_auc': valid_metrics['roc_auc'],
            'val_pr_auc': valid_metrics['pr_auc'],
        }
        history.append(row)
        print(pd.Series(row).to_string())

        improved = is_better_checkpoint(row, best_row, cfg)
        if improved:
            best_auc = row['val_auc']
            best_row = row.copy()
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(
                {
                    'state_dict': get_model_state_dict(model),
                    'cfg': asdict(cfg),
                    'best_auc': best_auc,
                    'best_epoch': best_epoch,
                    'run_name': cfg.name,
                },
                ckpt_path,
            )
            print(f'[BEST] saved checkpoint: {ckpt_path}')
        else:
            patience_counter += 1
            if patience_counter >= cfg.patience:
                print(f'Early stopping at epoch {epoch+1}; best_auc={best_auc:.4f}')
                break

    history_df = pd.DataFrame(history)
    history_df.to_csv(LOG_DIR / f'{cfg.name}_history.csv', index=False)

    elapsed_min = (time.time() - start) / 60
    summary = asdict(cfg)
    summary.update(best_row or {})
    summary['status'] = 'ok'
    summary.update(
        {
            'best_epoch': best_epoch,
            'best_val_auc': best_auc,
            'checkpoint': str(ckpt_path),
            'elapsed_min': elapsed_min,
        }
    )

    del model, optimizer, scheduler, scaler, criterion, train_loader, valid_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


def failed_summary(cfg: ExperimentConfig, exc: Exception) -> Dict:
    row = asdict(cfg)
    row.update({
        'status': 'failed',
        'error': f'{type(exc).__name__}: {exc}',
        'best_epoch': -1,
        'best_val_auc': -np.inf,
        'val_loss': np.inf,
        'val_accuracy': np.nan,
        'val_precision': np.nan,
        'val_recall': np.nan,
        'val_f1': np.nan,
        'val_pr_auc': np.nan,
        'checkpoint': '',
        'elapsed_min': np.nan,
    })
    return row


def run_many(configs: List[ExperimentConfig]) -> pd.DataFrame:
    rows = []
    for cfg in configs:
        try:
            rows.append(run_experiment(cfg))
        except Exception as exc:
            print(f'[FAILED] {cfg.name}: {type(exc).__name__}: {exc}')
            rows.append(failed_summary(cfg, exc))
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        pd.DataFrame(rows).to_csv(OUT_DIR / 'partial_summary.csv', index=False)
    return pd.DataFrame(rows)


## 9. Tuning plan builders

`minimal6` is the practical low-cost plan. `full11` is more complete but costs more GPU time.


In [ ]:
def with_name(cfg: ExperimentConfig, name: str, stage: str, **kwargs) -> ExperimentConfig:
    return replace(cfg, name=name, stage=stage, **kwargs)


def stage1_lr_configs(plan: str) -> List[ExperimentConfig]:
    configs = [
        with_name(BASE_CFG, 'run01_lr2e5_head1e4_512_mean', 'stage1_lr', lr_backbone=2e-5, lr_head=1e-4),
        with_name(BASE_CFG, 'run02_lr1e5_head5e5_512_mean', 'stage1_lr', lr_backbone=1e-5, lr_head=5e-5),
        with_name(BASE_CFG, 'run03_lr2e5_head5e5_512_mean', 'stage1_lr', lr_backbone=2e-5, lr_head=5e-5),
    ]
    if plan == 'full11':
        configs.append(with_name(BASE_CFG, 'run04_lr1e5_head1e4_512_mean', 'stage1_lr', lr_backbone=1e-5, lr_head=1e-4))
    return configs


def stage2_structure_configs(best_cfg: ExperimentConfig, plan: str) -> List[ExperimentConfig]:
    configs = [
        with_name(best_cfg, 'run_struct_512_cls', 'stage2_structure', max_length=512, pooling='cls'),
        with_name(best_cfg, 'run_struct_256_mean', 'stage2_structure', max_length=256, pooling='mean'),
    ]
    if plan == 'full11':
        configs.append(with_name(best_cfg, 'run_struct_256_cls', 'stage2_structure', max_length=256, pooling='cls'))
    return configs


def stage3_regularization_configs(best_cfg: ExperimentConfig, plan: str) -> List[ExperimentConfig]:
    configs = [
        with_name(best_cfg, 'run_reg_dropout02', 'stage3_regularization', dropout=0.2),
    ]
    if plan == 'full11':
        configs.extend(
            [
                with_name(best_cfg, 'run_reg_cosine', 'stage3_regularization', scheduler_type='cosine'),
                with_name(best_cfg, 'run_reg_label_smoothing005', 'stage3_regularization', label_smoothing=0.05),
                with_name(best_cfg, 'run_reg_class_weights', 'stage3_regularization', use_class_weights=True),
            ]
        )
    if INCLUDE_LLRD_RUN:
        configs.append(with_name(best_cfg, 'run_reg_llrd095', 'stage3_regularization', llrd_factor=0.95))
    return configs


def cfg_from_summary_row(row: pd.Series) -> ExperimentConfig:
    cfg_fields = ExperimentConfig.__dataclass_fields__.keys()
    kwargs = {k: row[k] for k in cfg_fields if k in row.index}
    int_fields = ['max_length', 'num_labels', 'epochs', 'batch_size', 'valid_batch_size', 'grad_accum', 'num_workers', 'patience']
    float_fields = ['dropout', 'max_grad_norm', 'lr_backbone', 'lr_head', 'llrd_factor', 'weight_decay', 'warmup_ratio', 'eps', 'beta1', 'beta2', 'min_delta', 'selection_auc_tolerance', 'selection_f1_tolerance', 'label_smoothing']
    bool_fields = ['use_amp', 'use_class_weights']
    for key in int_fields:
        if key in kwargs:
            kwargs[key] = int(kwargs[key])
    for key in float_fields:
        if key in kwargs:
            kwargs[key] = float(kwargs[key])
    def parse_bool(value):
        if isinstance(value, str):
            return value.strip().lower() in {'true', '1', 'yes'}
        return bool(value)

    for key in bool_fields:
        if key in kwargs:
            kwargs[key] = parse_bool(kwargs[key])
    return ExperimentConfig(**kwargs)


def choose_best(summary_df: pd.DataFrame) -> pd.Series:
    candidates = summary_df.copy()
    if 'status' in candidates.columns:
        candidates = candidates[candidates['status'].fillna('ok').eq('ok')]
    candidates = candidates[np.isfinite(pd.to_numeric(candidates['best_val_auc'], errors='coerce'))]
    if candidates.empty:
        raise RuntimeError('No successful tuning runs with finite validation AUC.')

    candidates = candidates.copy()
    candidates['best_val_auc'] = pd.to_numeric(candidates['best_val_auc'], errors='coerce')
    candidates['val_f1'] = pd.to_numeric(candidates['val_f1'], errors='coerce')
    candidates['val_loss'] = pd.to_numeric(candidates['val_loss'], errors='coerce')
    max_auc = candidates['best_val_auc'].max()
    near_best_auc = candidates[candidates['best_val_auc'] >= max_auc - BASE_CFG.selection_auc_tolerance]
    return near_best_auc.sort_values(['val_f1', 'val_loss', 'best_val_auc'], ascending=[False, True, False]).iloc[0]


## 10. Run staged tuning

Đổi `RUN_TUNING=True` để chạy thật. Notebook sẽ:

1. chạy stage 1,
2. chọn best learning rate,
3. chạy stage 2 từ best stage 1,
4. chọn best structure,
5. chạy stage 3 từ best stage 2,
6. lưu best config cuối cùng.


In [ ]:
all_results = []

if RUN_TUNING:
    if TUNING_PLAN not in {'minimal6', 'full11'}:
        raise ValueError("TUNING_PLAN must be 'minimal6' or 'full11'")

    stage1 = stage1_lr_configs(TUNING_PLAN)
    stage1_df = run_many(stage1)
    all_results.append(stage1_df)
    best_stage1_row = choose_best(stage1_df)
    best_stage1_cfg = cfg_from_summary_row(best_stage1_row)
    print('\nBest after Stage 1:')
    display(best_stage1_row.to_frame().T)

    stage2 = stage2_structure_configs(best_stage1_cfg, TUNING_PLAN)
    stage2_df = run_many(stage2)
    all_results.append(stage2_df)
    stage12_df = pd.concat([stage1_df, stage2_df], ignore_index=True)
    best_stage2_row = choose_best(stage12_df)
    best_stage2_cfg = cfg_from_summary_row(best_stage2_row)
    print('\nBest after Stage 2:')
    display(best_stage2_row.to_frame().T)

    stage3 = stage3_regularization_configs(best_stage2_cfg, TUNING_PLAN)
    stage3_df = run_many(stage3)
    all_results.append(stage3_df)

    final_summary = pd.concat(all_results, ignore_index=True)
    final_summary = final_summary.sort_values(['best_val_auc', 'val_f1', 'val_loss'], ascending=[False, False, True]).reset_index(drop=True)
    final_summary.to_csv(SUMMARY_PATH, index=False)

    best_row = final_summary.iloc[0]
    best_cfg = cfg_from_summary_row(best_row)
    BEST_CONFIG_PATH.write_text(json.dumps(asdict(best_cfg), indent=2, ensure_ascii=False), encoding='utf-8')

    print('\nFinal tuning summary:')
    display(final_summary)
    print('\nBest config:')
    print(json.dumps(asdict(best_cfg), indent=2, ensure_ascii=False))
else:
    print('RUN_TUNING=False: notebook chưa chạy tuning. Đổi thành True khi chạy trên GPU/Kaggle.')


## 11. Inspect existing summary

Cell này dùng sau khi đã chạy tuning, hoặc khi bạn mở lại notebook sau một phiên Kaggle trước đó.


In [ ]:
if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
    summary_df = summary_df.sort_values(['best_val_auc', 'val_f1', 'val_loss'], ascending=[False, False, True]).reset_index(drop=True)
    display(summary_df)
    print('Best run:')
    display(summary_df.head(1))
else:
    print(f'Chưa có summary: {SUMMARY_PATH}')


## 12. Promote best checkpoint for export/demo

After tuning, this cell copies the best checkpoint to `outputs/checkpoints/deberta_tuned_best.pt`. Notebook export demo sẽ ưu tiên checkpoint này nếu tồn tại.


In [ ]:
if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
    best_row = choose_best(summary_df)
    best_ckpt = Path(best_row['checkpoint'])
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Best checkpoint not found: {best_ckpt}')

    shutil.copy2(best_ckpt, PROMOTED_CKPT_PATH)
    BEST_CONFIG_PATH.write_text(
        json.dumps(cfg_from_summary_row(best_row).__dict__, indent=2, ensure_ascii=False),
        encoding='utf-8',
    )
    print(f'Promoted best checkpoint: {best_ckpt}')
    print(f'Copied to: {PROMOTED_CKPT_PATH}')
    print(f'Best config saved to: {BEST_CONFIG_PATH}')
else:
    print('Chưa có summary nên chưa promote checkpoint.')


## 12.1 Held-out test evaluation for best tuned checkpoint

Evaluate the selected tuning checkpoint once on the held-out test split. Do not use this result for hyperparameter selection.

In [ ]:
if not SUMMARY_PATH.exists():
    print(f'No tuning summary found at {SUMMARY_PATH}. Run tuning first.')
else:
    summary_df = pd.read_csv(SUMMARY_PATH)
    if summary_df.empty:
        print('Tuning summary is empty. Run tuning first.')
    else:
        best_row = choose_best(summary_df)
        best_cfg = cfg_from_summary_row(best_row)
        best_ckpt = Path(best_row.get('checkpoint_path', checkpoint_path_for(best_cfg)))
        if not best_ckpt.exists() and PROMOTED_CKPT_PATH.exists():
            best_ckpt = PROMOTED_CKPT_PATH

        if not best_ckpt.exists():
            print(f'Best checkpoint not found: {best_ckpt}')
        else:
            tokenizer = AutoTokenizer.from_pretrained(best_cfg.model_name)
            test_ds = AITextDataset(
                test_df[text_col].tolist(),
                test_df['label'].tolist(),
                tokenizer,
                best_cfg.max_length,
            )
            test_loader = DataLoader(
                test_ds,
                batch_size=best_cfg.valid_batch_size,
                shuffle=False,
                num_workers=best_cfg.num_workers,
                pin_memory=torch.cuda.is_available(),
                drop_last=False,
            )

            model = DeBERTaDetector(
                model_name=best_cfg.model_name,
                num_labels=best_cfg.num_labels,
                dropout=best_cfg.dropout,
                pooling=best_cfg.pooling,
            ).to(DEVICE)
            model = force_trainable_fp32(model)
            ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
            model.load_state_dict(ckpt['state_dict'])
            model.eval()

            criterion = make_criterion(best_cfg)
            test_metrics = evaluate(model, test_loader, criterion, best_cfg)

            test_pred_df = test_df[[text_col, 'label', 'split', 'fold']].copy()
            test_pred_df['prob_ai'] = test_metrics['probs']
            test_pred_df['pred'] = (test_pred_df['prob_ai'] >= 0.5).astype(int)

            test_metric_row = {
                'split': 'test',
                'checkpoint': str(best_ckpt),
                'run_name': best_row.get('name', ''),
                'best_epoch': best_row.get('best_epoch', ''),
                'max_length': best_cfg.max_length,
                'pooling': best_cfg.pooling,
                'dropout': best_cfg.dropout,
                'lr_backbone': best_cfg.lr_backbone,
                'lr_head': best_cfg.lr_head,
                'test_loss': test_metrics['loss'],
                'test_accuracy': test_metrics['accuracy'],
                'test_precision': test_metrics['precision'],
                'test_recall': test_metrics['recall'],
                'test_f1': test_metrics['f1'],
                'test_auc': test_metrics['roc_auc'],
                'test_pr_auc': test_metrics['pr_auc'],
            }
            test_metrics_df = pd.DataFrame([test_metric_row])

            pred_path = OUT_DIR / 'best_tuned_test_predictions.csv'
            metrics_path = OUT_DIR / 'best_tuned_test_metrics.csv'
            test_pred_df.to_csv(pred_path, index=False)
            test_metrics_df.to_csv(metrics_path, index=False)

            display(test_metrics_df)
            print(f'Saved test predictions: {pred_path}')
            print(f'Saved test metrics: {metrics_path}')

            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

## 13. Report table template

Bảng này là format nên đưa vào báo cáo sau khi chạy tuning.


In [ ]:
if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
    report_cols = [
        'name', 'stage', 'lr_backbone', 'lr_head', 'max_length', 'pooling',
        'dropout', 'scheduler_type', 'label_smoothing', 'use_class_weights',
        'best_epoch', 'val_loss', 'best_val_auc', 'val_f1', 'val_accuracy',
        'val_precision', 'val_recall',
    ]
    available_cols = [c for c in report_cols if c in summary_df.columns]
    report_df = summary_df[available_cols].sort_values('best_val_auc', ascending=False)
    display(report_df)
    report_df.to_csv(OUT_DIR / 'report_tuning_table.csv', index=False)
else:
    print('Chưa có summary để tạo bảng báo cáo.')


## 14. Training curves from logged histories

Load per-epoch CSV logs from all tuning runs and export comparison plots for the report. These plots are generated from local CSV logs, so CometML is optional.

In [ ]:
history_files = sorted(LOG_DIR.glob('*_history.csv'))

if not history_files:
    print(f'No history files found in {LOG_DIR}. Run tuning experiments first.')
else:
    history_frames = []
    for history_file in history_files:
        df_hist = pd.read_csv(history_file)
        if 'run_name' not in df_hist.columns:
            df_hist['run_name'] = history_file.stem.replace('_history', '')
        history_frames.append(df_hist)

    history_all = pd.concat(history_frames, ignore_index=True)
    history_path = OUT_DIR / 'all_epoch_history.csv'
    history_all.to_csv(history_path, index=False)

    plot_df = history_all.sort_values(['run_name', 'epoch'])

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for run_name, run_df in plot_df.groupby('run_name'):
        axes[0].plot(run_df['epoch'], run_df['train_loss'], marker='o', linewidth=1.5, label=f'{run_name} train')
        axes[1].plot(run_df['epoch'], run_df['val_loss'], marker='o', linewidth=1.5, label=f'{run_name} val')

    axes[0].set_title('Training loss by epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Train loss')
    axes[0].grid(alpha=0.3)

    axes[1].set_title('Validation loss by epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation loss')
    axes[1].grid(alpha=0.3)

    for ax in axes:
        ax.legend(fontsize=8, ncol=1)

    plt.tight_layout()
    loss_plot_path = OUT_DIR / 'tuning_loss_curves.png'
    plt.savefig(loss_plot_path, dpi=160, bbox_inches='tight')
    plt.show()

    metric_cols = [col for col in ['val_auc', 'val_f1'] if col in plot_df.columns]
    if metric_cols:
        fig, axes = plt.subplots(1, len(metric_cols), figsize=(7 * len(metric_cols), 5))
        if len(metric_cols) == 1:
            axes = [axes]

        for ax, metric in zip(axes, metric_cols):
            for run_name, run_df in plot_df.groupby('run_name'):
                ax.plot(run_df['epoch'], run_df[metric], marker='o', linewidth=1.5, label=run_name)
            ax.set_title(f'{metric} by epoch')
            ax.set_xlabel('Epoch')
            ax.set_ylabel(metric)
            ax.grid(alpha=0.3)
            ax.legend(fontsize=8)

        plt.tight_layout()
        metric_plot_path = OUT_DIR / 'tuning_metric_curves.png'
        plt.savefig(metric_plot_path, dpi=160, bbox_inches='tight')
        plt.show()

    display(history_all.head())
    print(f'Saved combined epoch history to: {history_path}')
    print(f'Saved loss plot to: {loss_plot_path}')
    if metric_cols:
        print(f'Saved metric plot to: {metric_plot_path}')

## 15. Best run training process for report

Create report-ready artifacts for the selected DeBERTa run: train/validation loss curve, validation metric curve, per-epoch metric table, and a compact checkpoint summary.

In [ ]:
if not SUMMARY_PATH.exists():
    print(f'No tuning summary found at {SUMMARY_PATH}. Run the tuning notebook first.')
else:
    summary_df = pd.read_csv(SUMMARY_PATH)
    if summary_df.empty:
        print('Tuning summary is empty. Run tuning experiments first.')
    else:
        best_row = choose_best(summary_df)
        best_run_name = best_row['name']
        history_file = LOG_DIR / f'{best_run_name}_history.csv'

        if not history_file.exists():
            print(f'No history file found for best run: {history_file}')
            print('Available history files:')
            for file in sorted(LOG_DIR.glob('*_history.csv')):
                print(' -', file.name)
        else:
            hist = pd.read_csv(history_file).sort_values('epoch').reset_index(drop=True)
            report_dir = OUT_DIR / 'report_training_process'
            report_dir.mkdir(parents=True, exist_ok=True)

            epoch_cols = [
                'epoch', 'train_loss', 'val_loss', 'val_auc', 'val_f1',
                'val_precision', 'val_recall', 'val_accuracy', 'val_pr_auc',
            ]
            epoch_cols = [col for col in epoch_cols if col in hist.columns]
            epoch_table = hist[epoch_cols].copy()
            epoch_table_path = report_dir / 'best_run_epoch_metrics.csv'
            epoch_table.to_csv(epoch_table_path, index=False)

            if 'best_epoch' in best_row and pd.notna(best_row['best_epoch']):
                best_epoch = int(best_row['best_epoch'])
            elif 'val_auc' in hist.columns:
                best_epoch = int(hist.loc[hist['val_auc'].idxmax(), 'epoch'])
            else:
                best_epoch = int(hist.loc[hist['val_loss'].idxmin(), 'epoch'])

            summary_items = {
                'best_run': best_run_name,
                'best_epoch': best_epoch,
                'checkpoint_path': best_row.get('checkpoint_path', ''),
                'elapsed_min': best_row.get('elapsed_min', ''),
                'lr_backbone': best_row.get('lr_backbone', ''),
                'lr_head': best_row.get('lr_head', ''),
                'max_length': best_row.get('max_length', ''),
                'pooling': best_row.get('pooling', ''),
                'dropout': best_row.get('dropout', ''),
                'scheduler_type': best_row.get('scheduler_type', ''),
                'label_smoothing': best_row.get('label_smoothing', ''),
                'use_class_weights': best_row.get('use_class_weights', ''),
                'best_val_loss': best_row.get('val_loss', ''),
                'best_val_auc': best_row.get('val_auc', ''),
                'best_val_f1': best_row.get('val_f1', ''),
                'best_val_precision': best_row.get('val_precision', ''),
                'best_val_recall': best_row.get('val_recall', ''),
            }
            summary_table = pd.DataFrame(
                [{'item': key, 'value': value} for key, value in summary_items.items()]
            )
            summary_path = report_dir / 'best_run_training_summary.csv'
            summary_table.to_csv(summary_path, index=False)

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(hist['epoch'], hist['train_loss'], marker='o', linewidth=2, label='Train loss')
            ax.plot(hist['epoch'], hist['val_loss'], marker='s', linewidth=2, label='Validation loss')
            ax.axvline(best_epoch, color='black', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Best epoch {best_epoch}')
            ax.set_title(f'Training and validation loss - {best_run_name}')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.grid(alpha=0.3)
            ax.legend()
            plt.tight_layout()
            loss_path = report_dir / 'best_run_loss_curve.png'
            plt.savefig(loss_path, dpi=180, bbox_inches='tight')
            plt.show()

            metric_cols = [col for col in ['val_auc', 'val_f1', 'val_precision', 'val_recall'] if col in hist.columns]
            if metric_cols:
                fig, ax = plt.subplots(figsize=(8, 5))
                for metric in metric_cols:
                    ax.plot(hist['epoch'], hist[metric], marker='o', linewidth=2, label=metric)
                ax.axvline(best_epoch, color='black', linestyle='--', linewidth=1.2, alpha=0.7, label=f'Best epoch {best_epoch}')
                ax.set_title(f'Validation metrics - {best_run_name}')
                ax.set_xlabel('Epoch')
                ax.set_ylabel('Score')
                ax.set_ylim(0, 1.02)
                ax.grid(alpha=0.3)
                ax.legend()
                plt.tight_layout()
                metrics_path = report_dir / 'best_run_validation_metrics.png'
                plt.savefig(metrics_path, dpi=180, bbox_inches='tight')
                plt.show()
            else:
                metrics_path = None

            display(summary_table)
            display(epoch_table)
            print(f'Saved summary: {summary_path}')
            print(f'Saved epoch metrics: {epoch_table_path}')
            print(f'Saved loss curve: {loss_path}')
            if metrics_path is not None:
                print(f'Saved validation metric curve: {metrics_path}')